In [19]:
import pandas as pd
import numpy as np

df = pd.read_excel('india_pollution_uncleaned.xlsx')
print("Shape:", df.shape)
print("Nulls before:\n", df.isnull().sum())

Shape: (34700, 64)
Nulls before:
 id                                   0
Station_Name                      5476
Date                                 0
Year                              4456
Month                             5778
                                  ... 
Public_Transport_Usage_Percent    4299
NCAP_Flag                         2170
River_Pollution_Rank              1824
Relative_Humidity_Pct             4356
Temperature_C                     6072
Length: 64, dtype: int64


In [20]:
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

Duplicates: 124


In [21]:
# Clean column names
df.columns = df.columns.str.strip()


In [23]:
text_cols = ['City', 'State', 'Season', 'AQI_Bucket', 'Month_Name',
             'River_Name', 'Station_Name', 'Water_Quality_Status',
             'Prominent_Pollutant', 'Industry_Sector']

for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()
    df[col] = df[col].replace('Nan', np.nan)


In [12]:
text_cols = ['City', 'State', 'Season', 'AQI_Bucket', 'Month_Name',
             'River_Name', 'Station_Name', 'Water_Quality_Status',
             'Prominent_Pollutant', 'Industry_Sector']

for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()
    df[col] = df[col].replace('Nan', np.nan)


In [24]:
df['AQI'] = (df['AQI'].astype(str)
              .str.replace('µg/m³', '', regex=False)
              .str.strip())
df['AQI'] = pd.to_numeric(df['AQI'], errors='coerce')

In [25]:
neg_cols = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx',
            'SO2', 'O3', 'CO', 'NH3', 'AQI']

for col in neg_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[df[col] < 0, col] = np.nan

In [26]:
# Any value above these thresholds is physically impossible
thresholds = {'PM2.5': 1000, 'PM10': 1500,
              'Temperature_C': 60, 'AQI': 500}

for col, limit in thresholds.items():
    df.loc[df[col] > limit, col] = np.nan

In [27]:
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df['Year'] = df['Year'].astype('Int64')

In [28]:
df['Date'] = df['Date'].astype(str).str.replace('99/99/9999', '')
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

In [29]:
bool_cols = ['PM25_Standard_Exceeded', 'PM10_Standard_Exceeded',
             'BOD_Standard_Exceeded', 'DO_Standard_Below']

yes_set = {'yes', 'YES', 'Yes', '1', 'True', 'TRUE', 'true'}
no_set  = {'no', 'NO', 'No', '0', 'False', 'FALSE', 'false'}

for col in bool_cols:
    s = df[col].astype(str).str.strip()
    df[col] = s.apply(lambda x:
        'Yes' if x in yes_set else ('No' if x in no_set else np.nan))

In [30]:
print("--- BEFORE filling nulls ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

num_cols = df.select_dtypes(include='number').columns
cat_cols = df.select_dtypes(include='object').columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("\n--- AFTER filling nulls ---")
print("Total nulls:", df.isnull().sum().sum())

--- BEFORE filling nulls ---
Station_Name                      5451
Date                               694
Year                              4437
Month                             5756
Month_Name                        5285
                                  ... 
Public_Transport_Usage_Percent    4277
NCAP_Flag                         2164
River_Pollution_Rank              1818
Relative_Humidity_Pct             4341
Temperature_C                     6882
Length: 63, dtype: int64

--- AFTER filling nulls ---
Total nulls: 694


In [32]:
# Fill remaining Date nulls with the most common date
most_common_date = df['Date'].mode()[0]
df['Date'] = df['Date'].fillna(most_common_date)

# Verify
print("Total nulls:", df.isnull().sum().sum())  # should print 0

Total nulls: 0


In [33]:
print("Final shape:", df.shape)
print("Nulls remaining:", df.isnull().sum().sum())
print(df.dtypes)

df.to_csv('india_pollution_cleaned.csv', index=False)
print("Saved!")

Final shape: (34576, 64)
Nulls remaining: 0
id                                         int64
Station_Name                              object
Date                              datetime64[ns]
Year                                       Int64
Month                                    float64
                                       ...      
Public_Transport_Usage_Percent           float64
NCAP_Flag                                 object
River_Pollution_Rank                     float64
Relative_Humidity_Pct                    float64
Temperature_C                            float64
Length: 64, dtype: object
Saved!
